# Model Evaluation on Test Set
Loads both fine-tuned XLM-R models (human and LLM annotations) and evaluates them on the human-labeled test set across EN, IT, SI.

In [1]:

!pip install transformers torch scikit-learn --quiet

## 1. Mount Drive and configure paths

In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b1-11y0s86c72o3i?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_lo

In [3]:
import os

# ── Paths ──
TEST_DATA_DIR   = '/content/drive/MyDrive/test_data'
MODEL_HUMAN_DIR = '/content/drive/MyDrive/Models/xlmr_human'
MODEL_LLM_DIR   = '/content/drive/MyDrive/Models/xlmr_llm'
OUTPUT_DIR      = '/content/drive/MyDrive/Models/evaluation'
LANGUAGES       = ['EN', 'IT', 'SI']
MAX_LENGTH      = 128
BATCH_SIZE      = 32   # larger batch fine for inference

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify models exist
for name, path in [('human', MODEL_HUMAN_DIR), ('llm', MODEL_LLM_DIR)]:
    exists = os.path.exists(os.path.join(path, 'model.safetensors'))
    print(f'Model {name}: {"✓ found" if exists else "✗ NOT FOUND"}')

# Verify test files exist
for lang in LANGUAGES:
    path = os.path.join(TEST_DATA_DIR, f'{lang}_test_set.csv')
    print(f'Test {lang}: {"✓ found" if os.path.exists(path) else "✗ NOT FOUND"}')

Model human: ✓ found
Model llm: ✓ found
Test EN: ✓ found
Test IT: ✓ found
Test SI: ✓ found


## 2. Load test data

In [4]:
import pandas as pd
import numpy as np

def load_test(lang):
    path = os.path.join(TEST_DATA_DIR, f'{lang}_test_set.csv')
    df = pd.read_csv(path)
    df = df[pd.to_numeric(df['label'], errors='coerce').isin([0,1,2,3])]
    df['label'] = df['label'].astype(int)
    df['lang'] = lang
    print(f'[{lang}] {len(df)} test comments | label dist: {df["label"].value_counts().sort_index().to_dict()}')
    return df

test_parts = [load_test(lang) for lang in LANGUAGES]
test_df    = pd.concat(test_parts, ignore_index=True)
print(f'\nTotal test comments: {len(test_df)}')

[EN] 21503 test comments | label dist: {0: 15934, 1: 183, 2: 5310, 3: 76}
[IT] 21070 test comments | label dist: {0: 15954, 1: 770, 2: 4082, 3: 264}
[SI] 20000 test comments | label dist: {0: 13273, 1: 285, 2: 6373, 3: 69}

Total test comments: 62573


## 3. Inference function

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze()
        }


def run_inference(model_dir, texts):
    """Load model from dir, run inference on texts, return predicted labels."""
    print(f'  Loading model from {model_dir}...')
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model     = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model     = model.to(device)
    model.eval()

    dataset = TestDataset(texts, tokenizer, MAX_LENGTH)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    all_preds = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
            preds          = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            if (i + 1) % 50 == 0:
                print(f'  Batch {i+1}/{len(loader)}')

    # Free GPU memory
    del model
    torch.cuda.empty_cache()
    return np.array(all_preds)

Device: cuda


## 4. Run inference with both models

In [7]:


texts = test_df['text'].tolist()

print('=== Running inference: HUMAN model ===')
preds_human = run_inference(MODEL_HUMAN_DIR, texts)

print('\n=== Running inference: LLM model ===')
preds_llm = run_inference(MODEL_LLM_DIR, texts)

test_df['pred_human'] = preds_human
test_df['pred_llm']   = preds_llm
print('\nInference complete.')

=== Running inference: HUMAN model ===
  Loading model from /content/drive/MyDrive/Models/xlmr_human...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Batch 50/1956
  Batch 100/1956
  Batch 150/1956
  Batch 200/1956
  Batch 250/1956
  Batch 300/1956
  Batch 350/1956
  Batch 400/1956
  Batch 450/1956
  Batch 500/1956
  Batch 550/1956
  Batch 600/1956
  Batch 650/1956
  Batch 700/1956
  Batch 750/1956
  Batch 800/1956
  Batch 850/1956
  Batch 900/1956
  Batch 950/1956
  Batch 1000/1956
  Batch 1050/1956
  Batch 1100/1956
  Batch 1150/1956
  Batch 1200/1956
  Batch 1250/1956
  Batch 1300/1956
  Batch 1350/1956
  Batch 1400/1956
  Batch 1450/1956
  Batch 1500/1956
  Batch 1550/1956
  Batch 1600/1956
  Batch 1650/1956
  Batch 1700/1956
  Batch 1750/1956
  Batch 1800/1956
  Batch 1850/1956
  Batch 1900/1956
  Batch 1950/1956

=== Running inference: LLM model ===
  Loading model from /content/drive/MyDrive/Models/xlmr_llm...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  Batch 50/1956
  Batch 100/1956
  Batch 150/1956
  Batch 200/1956
  Batch 250/1956
  Batch 300/1956
  Batch 350/1956
  Batch 400/1956
  Batch 450/1956
  Batch 500/1956
  Batch 550/1956
  Batch 600/1956
  Batch 650/1956
  Batch 700/1956
  Batch 750/1956
  Batch 800/1956
  Batch 850/1956
  Batch 900/1956
  Batch 950/1956
  Batch 1000/1956
  Batch 1050/1956
  Batch 1100/1956
  Batch 1150/1956
  Batch 1200/1956
  Batch 1250/1956
  Batch 1300/1956
  Batch 1350/1956
  Batch 1400/1956
  Batch 1450/1956
  Batch 1500/1956
  Batch 1550/1956
  Batch 1600/1956
  Batch 1650/1956
  Batch 1700/1956
  Batch 1750/1956
  Batch 1800/1956
  Batch 1850/1956
  Batch 1900/1956
  Batch 1950/1956

Inference complete.


## 5. Evaluate and compare

In [8]:
!pip install krippendorff --quiet

In [9]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import krippendorff

def compute_alpha(true, pred):
    return krippendorff.alpha(
        reliability_data=[true.tolist(), pred.tolist()],
        level_of_measurement='ordinal'
    )

def print_confusion(cm, title):
    labels = ['0:Approp', '1:Inapprop', '2:Offensive', '3:Violent']
    col_w = 12
    print(f'\n  {title}')
    print('  ' + ' ' * 14 + ''.join(f'{l:>{col_w}}' for l in labels))
    print('  ' + '-' * (14 + col_w * 4))
    for i, rl in enumerate(labels):
        print('  ' + f'{rl:<14}' + ''.join(f'{cm[i,j]:>{col_w}}' for j in range(4)))


def evaluate_model(df, pred_col, model_name):
    true   = df['label'].values
    pred   = df[pred_col].values
    f1     = f1_score(true, pred, average='macro', zero_division=0)
    acc    = accuracy_score(true, pred)
    alpha = compute_alpha(true, pred)
    cm     = confusion_matrix(true, pred, labels=[0,1,2,3])
    return {'model': model_name, 'macro_f1': f1, 'accuracy': acc, 'alpha': alpha, 'cm': cm}


print('=' * 65)
print('  MODEL EVALUATION ON HUMAN-LABELED TEST SET')
print('=' * 65)

summary_rows = []

for lang in LANGUAGES + ['ALL']:
    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]
    print(f'\n{"-" * 65}')
    print(f'  LANGUAGE: {lang}  (n={len(subset)})')
    print(f'{"-" * 65}')

    for pred_col, model_name in [('pred_human', 'Human-label model'), ('pred_llm', 'LLM-label model')]:
        res = evaluate_model(subset, pred_col, model_name)
        print(f'\n  {model_name}')
        print(f'    Macro F1 : {res["macro_f1"]:.4f}')
        print(f'    Accuracy : {res["accuracy"]:.4f}')
        print(f'    Alpha    : {res["alpha"]:.4f}')
        print_confusion(res['cm'], 'Confusion matrix (rows=true, cols=predicted)')
        summary_rows.append({'language': lang, 'model': model_name,
                              'macro_f1': res['macro_f1'],
                              'accuracy': res['accuracy'],
                              'alpha': res['alpha']})

print(f'\n{"=" * 65}')
print('  SUMMARY TABLE')
print(f'{"=" * 65}')
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

  MODEL EVALUATION ON HUMAN-LABELED TEST SET

-----------------------------------------------------------------
  LANGUAGE: EN  (n=21503)
-----------------------------------------------------------------

  Human-label model
    Macro F1 : 0.5630
    Accuracy : 0.8312
    Alpha    : 0.6183

  Confusion matrix (rows=true, cols=predicted)
                    0:Approp  1:Inapprop 2:Offensive   3:Violent
  --------------------------------------------------------------
  0:Approp             13469          56        2373          36
  1:Inapprop              43          58          80           2
  2:Offensive            881          60        4314          55
  3:Violent               13           0          30          33

  LLM-label model
    Macro F1 : 0.5110
    Accuracy : 0.8126
    Alpha    : 0.5127

  Confusion matrix (rows=true, cols=predicted)
                    0:Approp  1:Inapprop 2:Offensive   3:Violent
  --------------------------------------------------------------
  0:Appr

## 6. Save results

In [10]:
# Save summary table
summary_path = os.path.join(OUTPUT_DIR, 'evaluation_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f'Summary saved to {summary_path}')

# Save full predictions
preds_path = os.path.join(OUTPUT_DIR, 'test_predictions.csv')
test_df.to_csv(preds_path, index=False)
print(f'Full predictions saved to {preds_path}')

Summary saved to /content/drive/MyDrive/Models/evaluation/evaluation_summary.csv
Full predictions saved to /content/drive/MyDrive/Models/evaluation/test_predictions.csv


In [ ]:
import pandas as pd

TEST_PREDICTIONS_PATH = '/content/drive/MyDrive/Models/evaluation/test_predictions.csv'
test_predictions = pd.read_csv(TEST_PREDICTIONS_PATH)
print(f'Loaded {len(test_predictions)} rows from {TEST_PREDICTIONS_PATH}')

## Bootstrapping

In [11]:
from sklearn.metrics import f1_score, accuracy_score
import krippendorff
import numpy as np
import pandas as pd

In [12]:
def macro_f1(y, pred):
    return f1_score(y, pred, average="macro")

def accuracy(y, pred):
    return accuracy_score(y, pred)

def alpha_metric(y, pred):
    return krippendorff.alpha(
        reliability_data=[y.tolist(), pred.tolist()],
        level_of_measurement='ordinal'
    )

In [13]:
METRICS = {
    "Macro F1": macro_f1,
    "Accuracy": accuracy,
    "Alpha": alpha_metric
}

In [14]:
def paired_bootstrap(
    y_true,
    pred_human,
    pred_llm,
    metric_fn,
    n_boot=5000,
    random_state=42
):

    rng = np.random.default_rng(random_state)

    n = len(y_true)

    observed = (
        metric_fn(y_true, pred_human)
        - metric_fn(y_true, pred_llm)
    )

    diffs = np.empty(n_boot)

    for b in range(n_boot):

        idx = rng.integers(0, n, n)

        yt = y_true[idx]
        ph = pred_human[idx]
        pl = pred_llm[idx]

        diffs[b] = (
            metric_fn(yt, ph)
            - metric_fn(yt, pl)
        )

    return {
        "observed": observed,
        "median": np.median(diffs),
        "ci_low": np.percentile(diffs,2.5),
        "ci_high": np.percentile(diffs,97.5),
        "distribution": diffs
    }

In [15]:
bootstrap_rows = []

for lang in LANGUAGES + ['ALL']:

    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]

    y_true = subset['label'].to_numpy()
    pred_human = subset['pred_human'].to_numpy()
    pred_llm = subset['pred_llm'].to_numpy()

    for metric_name, metric_fn in METRICS.items():

        result = paired_bootstrap(
            y_true,
            pred_human,
            pred_llm,
            metric_fn,
            n_boot=5000,
            random_state=42
        )

        bootstrap_rows.append({
            "language": lang,
            "metric": metric_name,
            "observed": result["observed"],
            "median": result["median"],
            "ci_low": result["ci_low"],
            "ci_high": result["ci_high"]
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)

display(bootstrap_df)

bootstrap_df.to_csv(
    "bootstrap_results.csv",
    index=False
)

,language,metric,observed,median,ci_low,ci_high
0,EN,Macro F1,0.052049,0.051431,0.026473,0.078077
1,EN,Accuracy,0.018602,0.018556,0.012835,0.024276
2,EN,Alpha,0.105528,0.105570,0.091307,0.119958
3,IT,Macro F1,0.083119,0.083142,0.066745,0.099709
4,IT,Accuracy,0.056763,0.056739,0.051495,0.061699
5,IT,Alpha,0.075991,0.076122,0.063574,0.088325
6,SI,Macro F1,0.156753,0.156689,0.128181,0.185179
7,SI,Accuracy,0.068000,0.068100,0.062050,0.074000
8,SI,Alpha,0.181857,0.181950,0.167774,0.195865
9,ALL,Macro F1,0.108553,0.108530,0.096691,0.120662


In [16]:
from sklearn.metrics import classification_report

rows = []

for lang in LANGUAGES + ['ALL']:

    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]

    for pred_col, model_name in [
        ('pred_human', 'Human'),
        ('pred_llm', 'LLM')
    ]:

        report = classification_report(
            subset['label'],
            subset[pred_col],
            target_names=[
                'Appropriate',
                'Inappropriate',
                'Offensive',
                'Violent'
            ],
            output_dict=True,
            zero_division=0
        )

        row = {
            "Language": lang,
            "Model": model_name,
            "Appropriate": report["Appropriate"]["f1-score"],
            "Inappropriate": report["Inappropriate"]["f1-score"],
            "Offensive": report["Offensive"]["f1-score"],
            "Violent": report["Violent"]["f1-score"],
            "Macro F1": report["macro avg"]["f1-score"],
        }

        rows.append(row)

per_class_df = pd.DataFrame(rows)
display(per_class_df)

,Language,Model,Appropriate,Inappropriate,Offensive,Violent,Macro F1
0,EN,Human,0.887871,0.324930,0.712646,0.326733,0.563045
1,EN,LLM,0.888311,0.280315,0.583689,0.291667,0.510996
2,IT,Human,0.912224,0.600612,0.628542,0.512235,0.663403
3,IT,LLM,0.878302,0.475817,0.558321,0.408696,0.580284
4,SI,Human,0.865736,0.468803,0.709677,0.363636,0.601963
5,SI,LLM,0.843205,0.263644,0.497825,0.176166,0.445210
6,ALL,Human,0.890000,0.532302,0.691291,0.452991,0.641646
7,ALL,LLM,0.870843,0.384373,0.545914,0.331242,0.533093


In [17]:
rows_weighted = []

for lang in LANGUAGES + ['ALL']:
    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]

    for pred_col, model_name in [
        ('pred_human', 'Human'),
        ('pred_llm', 'LLM')
    ]:
        report = classification_report(
            subset['label'],
            subset[pred_col],
            target_names=['Appropriate', 'Inappropriate', 'Offensive', 'Violent'],
            output_dict=True,
            zero_division=0
        )

        rows_weighted.append({
            "Language": lang,
            "Model": model_name,
            "Weighted F1": report["weighted avg"]["f1-score"],
            "Macro F1": report["macro avg"]["f1-score"],
        })

weighted_df = pd.DataFrame(rows_weighted)
display(weighted_df)

,Language,Model,Weighted F1,Macro F1
0,EN,Human,0.837826,0.563045
1,EN,LLM,0.805804,0.510996
2,IT,Human,0.840865,0.663403
3,IT,LLM,0.795717,0.580284
4,SI,Human,0.808620,0.601963
5,SI,LLM,0.722590,0.445210
6,ALL,Human,0.830003,0.641646
7,ALL,LLM,0.775827,0.533093


In [18]:
import krippendorff
import numpy as np

LANGUAGES = ['EN', 'IT', 'SI']

print("Alpha between Human-label model and LLM-label model predictions")
print("=" * 55)

for lang in LANGUAGES + ['ALL']:
    subset = test_df if lang == 'ALL' else test_df[test_df['lang'] == lang]

    alpha = krippendorff.alpha(
        reliability_data=[
            subset['pred_human'].tolist(),
            subset['pred_llm'].tolist()
        ],
        level_of_measurement='ordinal'
    )
    print(f"{lang:>4}: alpha = {alpha:.4f}")

Alpha between Human-label model and LLM-label model predictions
  EN: alpha = 0.5682
  IT: alpha = 0.6667
  SI: alpha = 0.5299
 ALL: alpha = 0.5885


In [ ]:
from statsmodels.stats.multitest import multipletests

reject_holm, pvals_holm, _, _ = multipletests(pvals, method='holm')
holm_results = pd.DataFrame({
    'pval': pvals,
    'pval_holm': pvals_holm,
    'reject_holm': reject_holm,
})
holm_results